In [1]:
!pip install xgboost -q

# ----------------------------
# 1) Imports
# ----------------------------
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score

# ----------------------------
# 2) Load dataset
# ----------------------------
data = load_breast_cancer()
X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Number of classes:", len(np.unique(y)))
print("Class names:", data.target_names)

# ----------------------------
# 3) Train / Test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ----------------------------
# 4) Scaling for SVM / SVC
# ----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ----------------------------
# 5) Define models
# ----------------------------

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

# SVM with RBF kernel (SVC)
svm_rbf_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=42
)

# SVM with linear kernel (SVC as linear SVM)
svm_linear_model = SVC(
    kernel="linear",
    C=1.0,
    random_state=42
)

# XGBoost classifier
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

# ----------------------------
# 6) Train models
# ----------------------------
rf_model.fit(X_train, y_train)
svm_rbf_model.fit(X_train_scaled, y_train)
svm_linear_model.fit(X_train_scaled, y_train)
xgb_model.fit(X_train, y_train)

# ----------------------------
# 7) Evaluate models
# ----------------------------
rf_pred = rf_model.predict(X_test)
svm_rbf_pred = svm_rbf_model.predict(X_test_scaled)
svm_linear_pred = svm_linear_model.predict(X_test_scaled)
xgb_pred = xgb_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
svm_rbf_acc = accuracy_score(y_test, svm_rbf_pred)
svm_linear_acc = accuracy_score(y_test, svm_linear_pred)
xgb_acc = accuracy_score(y_test, xgb_pred)

# ----------------------------
# 8) Show results in a small table
# ----------------------------
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "SVM (RBF kernel - SVC)",
        "SVM (Linear kernel - SVC)",
        "XGBoost"
    ],
    "Accuracy": [
        rf_acc,
        svm_rbf_acc,
        svm_linear_acc,
        xgb_acc
    ]
})

print("\nModel performance on test set:")
print(results.sort_values("Accuracy", ascending=False).reset_index(drop=True))

Dataset shape: (569, 30)
Number of classes: 2
Class names: ['malignant' 'benign']

Model performance on test set:
                       Model  Accuracy
0     SVM (RBF kernel - SVC)  0.982456
1  SVM (Linear kernel - SVC)  0.973684
2              Random Forest  0.956140
3                    XGBoost  0.947368
